# Statistics

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
    PLOT_IMPORT_ERROR = ""
except Exception as exc:
    plt = None
    MATPLOTLIB_AVAILABLE = False
    PLOT_IMPORT_ERROR = str(exc)

if MATPLOTLIB_AVAILABLE:
    try:
        from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
        HAS_3D = True
    except Exception:
        HAS_3D = False
else:
    HAS_3D = False

RESULTS_PATH = Path("results/results.csv")


def parse_results(csv_path):
    rows = []

    with csv_path.open(newline="") as fh:
        reader = csv.DictReader(fh)
        required_fields = {
            "N",
            "P",
            "HASH",
            "EXEC_TYPE",
            "CHECKSUM",
            "THROUGHPUT",
            "PARTITION_TIME",
            "GLOBAL_TIME",
        }

        if reader.fieldnames is None:
            raise ValueError(f"{csv_path} is empty or missing the CSV header")

        missing_fields = required_fields - set(reader.fieldnames)
        if missing_fields:
            raise ValueError(
                f"{csv_path} is missing required columns: {sorted(missing_fields)}"
            )

        for row in reader:
            rows.append(
                {
                    "N": int(row["N"]),
                    "P": int(row["P"]),
                    "hash": row["HASH"],
                    "exec_type": row["EXEC_TYPE"],
                    "checksum": row["CHECKSUM"],
                    "throughput": float(row["THROUGHPUT"]),
                    "partition_time": float(row["PARTITION_TIME"]),
                    "global_time": float(row["GLOBAL_TIME"]),
                }
            )

    return rows


if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Results file not found: {RESULTS_PATH}. Run the benchmarks first."
    )

rows = parse_results(RESULTS_PATH)
if not rows:
    raise ValueError("results/results.csv exists but does not contain benchmark rows")

exec_types = sorted({row["exec_type"] for row in rows})
hashes = sorted({row["hash"] for row in rows})
n_values = sorted({row["N"] for row in rows})
p_values = sorted({row["P"] for row in rows})

print(f"Loaded {len(rows)} runs from {RESULTS_PATH}.")
print(f"Execution types: {exec_types}")
print(f"Hashes: {hashes}")
print(f"N values: {n_values}")
print(f"P values: {p_values}")


In [ ]:
checksum_groups = defaultdict(set)

for row in rows:
    checksum_groups[(row["N"], row["P"], row["hash"])].add(row["checksum"])

inconsistent = {
    key: values for key, values in checksum_groups.items() if len(values) > 1
}

if inconsistent:
    print("Checksum mismatch found:")
    for (N, P, hash_name), values in sorted(inconsistent.items()):
        print(f"  N={N}, P={P}, hash={hash_name}: {sorted(values)}")
else:
    print("Checksums are coherent across implementations for each (N, P, hash).")


In [ ]:
if not MATPLOTLIB_AVAILABLE:
    print(f"Plotting skipped: matplotlib is not available ({PLOT_IMPORT_ERROR}).")
else:
    color_map = plt.get_cmap("tab10")

    def plot_metric(metric_key, axis_label, title_prefix):
        if len(n_values) > 1 and len(p_values) > 1 and HAS_3D:
            for hash_name in hashes:
                fig = plt.figure(figsize=(10, 7))
                ax = fig.add_subplot(111, projection="3d")

                for idx, exec_type in enumerate(exec_types):
                    subset = [
                        row
                        for row in rows
                        if row["hash"] == hash_name and row["exec_type"] == exec_type
                    ]
                    if not subset:
                        continue

                    subset.sort(key=lambda row: (row["N"], row["P"]))
                    ax.scatter(
                        [row["N"] for row in subset],
                        [row["P"] for row in subset],
                        [row[metric_key] for row in subset],
                        label=exec_type,
                        color=color_map(idx % 10),
                        s=60,
                    )

                ax.set_title(f"{title_prefix} by N and P ({hash_name})")
                ax.set_xlabel("N")
                ax.set_ylabel("P")
                ax.set_zlabel(axis_label)
                ax.legend()
                plt.show()
        else:
            for hash_name in hashes:
                for N in n_values:
                    fig, ax = plt.subplots(figsize=(9, 5))
                    plotted = False

                    for idx, exec_type in enumerate(exec_types):
                        subset = [
                            row
                            for row in rows
                            if row["hash"] == hash_name
                            and row["N"] == N
                            and row["exec_type"] == exec_type
                        ]
                        if not subset:
                            continue

                        subset.sort(key=lambda row: row["P"])
                        ax.plot(
                            [row["P"] for row in subset],
                            [row[metric_key] for row in subset],
                            marker="o",
                            label=exec_type,
                            color=color_map(idx % 10),
                        )
                        plotted = True

                    if not plotted:
                        plt.close(fig)
                        continue

                    ax.set_title(f"{title_prefix} vs P for N={N} ({hash_name})")
                    ax.set_xlabel("P")
                    ax.set_ylabel(axis_label)
                    ax.set_xticks(sorted({row["P"] for row in rows if row["N"] == N}))
                    ax.grid(True, linestyle="--", alpha=0.4)
                    ax.legend()
                    plt.show()

    plot_metric("partition_time", "partition time [s]", "Partition Time")
    plot_metric("throughput", "throughput [elem/s]", "Throughput")
